# 02 - Data Cleaning and Feature Encoding

Continuing from the first notebook. Here I want to look closer at individual features vs pass/fail, deal with the categorical columns, and decide what to do about G1/G2.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

df = pd.read_csv('../data/raw/student-mat.csv', sep=';')
df['pass'] = (df['G3'] >= 10).astype(int)
df.shape

## The G1/G2 problem

G1 and G2 are the first and second period grades. The final grade G3 is basically built on top of these, so if I include them as features the model will just learn "G1 and G2 are high -> G3 is high" which is kind of cheating - it's not really predicting from the *causes* of good performance, just from earlier grades of the same thing.

Since the whole point of this project is to see which real-life factors (family, study habits, etc.) matter, I'm dropping G1 and G2 and only keeping the social/demographic/school features as predictors.

In [ ]:
df_model = df.drop(columns=['G1', 'G2', 'G3'])
df_model.columns.tolist()

## Quick look at a few features I expect to matter

Before encoding everything, let's just eyeball a few features against pass/fail.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

sns.boxplot(data=df, x='pass', y='studytime', ax=axes[0,0])
axes[0,0].set_title('Study time vs Pass/Fail')

sns.boxplot(data=df, x='pass', y='absences', ax=axes[0,1])
axes[0,1].set_title('Absences vs Pass/Fail')

sns.countplot(data=df, x='famsup', hue='pass', ax=axes[1,0])
axes[1,0].set_title('Family Support vs Pass/Fail')

sns.boxplot(data=df, x='pass', y='failures', ax=axes[1,1])
axes[1,1].set_title('Past Failures vs Pass/Fail')

plt.tight_layout()
plt.show()

Past failures looks like it might be a strong one just from the boxplot - makes sense honestly, if you've failed a class before you're more likely to struggle again. Will confirm this properly with a hypothesis test in the next notebook.

## Encoding categorical columns

Some columns are yes/no (binary) and some have more categories (like Mjob, Fjob, reason). For the binary ones I'll just map yes/no to 1/0. For the multi-category ones I'll use one-hot encoding (pd.get_dummies) so the model doesn't assume an order that isn't there.

In [ ]:
binary_cols = ['schoolsup', 'famsup', 'paid', 'activities', 'nursery',
               'higher', 'internet', 'romantic']

for col in binary_cols:
    df_model[col] = df_model[col].map({'yes': 1, 'no': 0})

# these two are also binary but not yes/no
df_model['school'] = df_model['school'].map({'GP': 1, 'MS': 0})
df_model['sex'] = df_model['sex'].map({'F': 1, 'M': 0})
df_model['address'] = df_model['address'].map({'U': 1, 'R': 0})
df_model['famsize'] = df_model['famsize'].map({'GT3': 1, 'LE3': 0})
df_model['Pstatus'] = df_model['Pstatus'].map({'T': 1, 'A': 0})

df_model.head()

In [ ]:
# the rest have more than 2 categories, one-hot encode those
multi_cat_cols = ['Mjob', 'Fjob', 'reason', 'guardian']

df_model = pd.get_dummies(df_model, columns=multi_cat_cols, drop_first=True)
df_model.shape

In [ ]:
# quick sanity check - make sure nothing is left as text/object type
df_model.dtypes.value_counts()

Everything's numeric now (bool columns from get_dummies count as fine for the model, will just cast them to int before training).

## Next steps

- hypothesis testing on a few of the strongest-looking features (failures, absences, studytime)
- train/test split and scaling
- then start on the from-scratch logistic regression